In [ ]:
import numpy as np
import pandas as pd
import random
from typing import Dict, Tuple, List

def get_alpha_range_for_density(density_level: str) -> Tuple[float, float]:
    """
    根据密度水平返回合适的alpha范围
    low density: alpha大 -> 间隔短 -> 每个csem任务少
    high density: alpha小 -> 间隔长 -> 每个csem任务多
    """
    if density_level == 'low':
        return (0.005, 0.01)   # 间隔短，任务少
    elif density_level == 'high':
        return (0.001, 0.003)  # 间隔长，任务多
    else:
        raise ValueError(f"Unknown density_level: {density_level}")

def generate_alpha_values_periodic(N: int, random_seed: int, 
                                 base_range: Tuple[float, float], 
                                 noise_ratio: float = 0.05) -> List[float]:
    """高周期性：所有csem alpha围绕基准微小扰动"""
    random.seed(random_seed)
    base_alpha = random.uniform(*base_range)
    alphas = []
    for i in range(N):
        noise = random.uniform(1 - noise_ratio, 1 + noise_ratio)
        alphas.append(round(base_alpha * noise, 6))
    return alphas

def generate_alpha_values_aperiodic(N: int, random_seed: int, 
                                  alpha_range: Tuple[float, float]) -> List[float]:
    """低周期性：alpha完全随机大范围"""
    random.seed(random_seed)
    return [round(random.uniform(*alpha_range), 6) for _ in range(N)]

def assign_offsets_for_clustering(N: int, T: int, random_seed: int, 
                                cluster_level: str, num_clusters: int = 3) -> List[int]:
    """
    控制聚类倾向：csem起始时间偏移
    uniform: 均匀分散在[0, T/2]
    clustered: 分num_clusters个时间点附近爆发
    """
    random.seed(random_seed + 2)
    offsets = [0] * N

    if cluster_level == 'uniform':
        # 均匀分散
        for i in range(N):
            offsets[i] = random.randint(0, T // 2)
    elif cluster_level == 'clustered':
        # 制造3个时间簇
        cluster_centers = [T // 4, T // 2, 3 * T // 4]
        cluster_sigma = 5  # ±5天聚堆
        for i in range(N):
            center = random.choice(cluster_centers)
            offset = max(0, min(T-1, int(random.gauss(center, cluster_sigma))))
            offsets[i] = offset
    else:
        raise ValueError(f"Unknown cluster_level: {cluster_level}")
    
    return offsets

def get_N_for_density(density_level: str, base_N: int = 90) -> int:
    """根据密度返回csem数量"""
    if density_level == 'low':
        return 30  # 稀疏
    elif density_level == 'high':
        return 150  # 密集
    else:
        raise ValueError(f"Unknown density_level: {density_level}")

def generate_maintenance_plan_scenario(
    density_level: str = 'low',
    periodicity_level: str = 'periodic',
    cluster_level: str = 'uniform',
    T: int = 180,
    random_seed: int = 42,
    initial_p_range: Tuple[float, float] = (3.2, 3.5),
    base_N: int = 90
) -> Tuple[pd.DataFrame, List[float]]:
    """
    🔥 DOE场景版生成函数：完全控制密度、周期性、聚类倾向
    
    Args:
        density_level: 'low'/'high' - 控制任务密度
        periodicity_level: 'periodic'/'aperiodic' - 控制周期性(CV)
        cluster_level: 'uniform'/'clustered' - 控制任务时空聚类
        T: 总天数
        random_seed: 随机种子
        base_N: 基准csem数
    """
    # 1) 根据密度决定N和alpha范围
    N = get_N_for_density(density_level, base_N)
    alpha_base_range = get_alpha_range_for_density(density_level)
    
    # 2) 根据周期性生成alpha
    if periodicity_level == 'periodic':
        alpha_values = generate_alpha_values_periodic(N, random_seed, alpha_base_range)
    else:
        alpha_values = generate_alpha_values_aperiodic(N, random_seed, alpha_base_range)
    
    # 3) 初始压力（保持原逻辑）
    random.seed(random_seed + 1)
    initial_p_values = [round(random.uniform(*initial_p_range), 2) for _ in range(N)]
    
    # 4) 根据聚类生成起始偏移
    offsets = assign_offsets_for_clustering(N, T, random_seed, cluster_level)
    
    # 5) 生成任务（保持你原有的物理逻辑）
    columns = ['INDEX', 'CSEM', 'INITIALP', 'OPERATION', 'DATE']
    data = []
    index = 0
    
    for caisson in range(N):
        current_p = initial_p_values[caisson]
        total_days = offsets[caisson]  # ★ 从offset开始控制聚类
        alpha = alpha_values[caisson]
        
        while total_days < T:
            time_to_3_2 = int((current_p - 3.2) / alpha)
            if total_days + time_to_3_2 >= T:
                break
            
            date = total_days + time_to_3_2
            data.append([index, caisson, initial_p_values[caisson], 'x', date])
            index += 1
            total_days = date + 1  # ★ 复位到下次维修起点
            current_p = 3.5
    
    maintenance_plan = pd.DataFrame(data, columns=columns)
    return maintenance_plan, alpha_values

# 🔥 场景特征验证函数（DOE必备）
def verify_scenario_features(df: pd.DataFrame, T: int, window_size: int = 10) -> Dict:
    """验证生成的场景特征"""
    # 密度：任务数/总天数
    density = len(df) / T
    
    # 周期性：各csem间隔CV平均值（低CV=高周期性）
    cv_dict = {}
    for csem in df['CSEM'].unique():
        csem_tasks = df[df['CSEM'] == csem].sort_values('DATE')
        if len(csem_tasks) < 2:
            cv_dict[csem] = 0
            continue
        intervals = np.diff(csem_tasks['DATE'].values)
        cv = np.std(intervals) / np.mean(intervals) if np.mean(intervals) > 0 else 0
        cv_dict[csem] = cv
    periodicity_cv = np.mean(list(cv_dict.values()))
    
    # 聚类倾向：窗口内平均任务数/随机预期
    daily_counts = df.groupby('DATE').size()
    window_tasks = []
    for start in range(0, int(df['DATE'].max()) - window_size + 1):
        mask = (df['DATE'] >= start) & (df['DATE'] <= start + window_size - 1)
        window_tasks.append(len(df[mask]))
    avg_window = np.mean(window_tasks) if window_tasks else 0
    random_expected = len(df) * window_size / (df['DATE'].max() + 1)
    clustering_tendency = avg_window / random_expected if random_expected > 0 else 1.0
    
    return {
        'density': round(density, 3),
        'periodicity_cv': round(periodicity_cv, 3),
        'clustering_tendency': round(clustering_tendency, 3),
        'total_tasks': len(df),
        'unique_csem': df['CSEM'].nunique()
    }

# 🔥 使用示例：直接生成8个DOE场景
if __name__ == "__main__":
    T = 180
    scenarios = [
        ('low', 'periodic', 'uniform'),
        ('low', 'periodic', 'clustered'),
        ('low', 'aperiodic', 'uniform'),
        ('low', 'aperiodic', 'clustered'),
        ('high', 'periodic', 'uniform'),
        ('high', 'periodic', 'clustered'),
        ('high', 'aperiodic', 'uniform'),
        ('high', 'aperiodic', 'clustered'),
    ]
    
    for i, (density, periodicity, clustering) in enumerate(scenarios, 1):
        print(f"\n=== 实验 {i}: {density}-{periodicity}-{clustering} ===")
        df, alphas = generate_maintenance_plan_scenario(
            density_level=density,
            periodicity_level=periodicity,
            cluster_level=clustering,
            T=T,
            random_seed=42
        )
        features = verify_scenario_features(df, T)
        print(f"特征: {features}")
        print(f"df shape: {df.shape}")
        print(df.head())


In [ ]:
# if run three models together for DOE
import time
import pandas as pd
from typing import Dict, List, Any
from concurrent.futures import ProcessPoolExecutor  # 并行加速

# 🔥 假设你已经有这些函数（从之前代码）
# - run_clustering_analysis(params, random_seed) → result_dict
# - calculate_detailed_stats(result_dict, params) → stats_dict
# - PARAMS全局参数

# 🔥 策略1：最小化组数ILP（修改你的策略2）
def strategy1_min_groups(params, random_seed, df):
    """策略1: 最小化组数"""
    # 复制你的clustering_CR，但目标改为 min y.sum()
    # 这里简化：返回你的run_clustering_analysis但修改目标函数
    result_dict = run_clustering_analysis(params, random_seed)
    # 修改obj = y.sum(), GRB.MINIMIZE
    stats = calculate_detailed_stats(result_dict, params)
    stats['strategy'] = 'ILP_MinGroups'
    return stats

# 🔥 策略2：你的成本最大化（已存在）
def strategy2_max_savings(params, random_seed, df):
    """策略2: 最大化成本节约（你的原版）"""
    result_dict = run_clustering_analysis(params, random_seed)
    stats = calculate_detailed_stats(result_dict, params)
    stats['strategy'] = 'ILP_MaxSavings'
    return stats

# 🔥 策略3：3天规则启发式
def strategy3_three_day_rule(df, window_size=10, Cmax=20):
    """策略3: 每3天窗口任务尽量放第1天"""
    df = df.copy()
    df['CENTER'] = df['DATE']  # 初始
    
    # 按DATE分组每3天窗口
    df_sorted = df.sort_values('DATE')
    current_window_start = 0
    
    while current_window_start < df['DATE'].max():
        window_end = min(current_window_start + 2, df['DATE'].max())
        window_tasks = df_sorted[
            (df_sorted['DATE'] >= current_window_start) & 
            (df_sorted['DATE'] <= window_end)
        ]
        
        if window_tasks.empty:
            break
            
        # 第1天尽量多（≤Cmax）
        day1_tasks = window_tasks.head(Cmax)
        day1_center = current_window_start
        
        # 剩余放第2天或第3天
        remaining = window_tasks.iloc[Cmax:]
        day2_center = current_window_start + 1 if len(remaining) > 0 else None
        
        # 更新CENTER
        df.loc[df['INDEX'].isin(day1_tasks['INDEX']), 'CENTER'] = day1_center
        if day2_center and len(remaining) > 0:
            df.loc[df['INDEX'].isin(remaining['INDEX']), 'CENTER'] = day2_center
            
        current_window_start += 3
    
    # 计算指标（复用你的函数）
    params = PARAMS.copy()
    leakage_after = calculate_gas_leakage(df, params['T'], use_center=True)
    # ... 其他指标计算
    stats = calculate_detailed_stats({'final_df': df}, params)  # 适配
    stats['strategy'] = 'ThreeDayRule'
    stats['total_clustering_cpu_s'] = 0.01  # 启发式很快
    return stats

# 🔥 单实例实验
def run_single_experiment(scenario_params: Dict, seed: int) -> List[Dict[str, Any]]:
    """运行一个场景+种子的3种策略"""
    density, periodicity, clustering = (
        scenario_params['density'], 
        scenario_params['periodicity'], 
        scenario_params['cluster_level']
    )
    
    # 生成场景数据
    df, alpha_vals = generate_maintenance_plan_scenario(
        density_level=density,
        periodicity_level=periodicity,
        cluster_level=clustering,
        T=PARAMS['T'],
        random_seed=seed
    )
    
    # 验证特征
    features = verify_scenario_features(df, PARAMS['T'])
    
    results = []
    
    # 策略1
    stats1 = strategy1_min_groups(PARAMS, seed, df)
    stats1.update(features)
    stats1['seed'] = seed
    stats1['scenario_id'] = scenario_params['id']
    results.append(stats1)
    
    # 策略2（你的原版）
    stats2 = strategy2_max_savings(PARAMS, seed, df)
    stats2.update(features)
    stats2['seed'] = seed
    stats2['scenario_id'] = scenario_params['id']
    results.append(stats2)
    
    # 策略3
    stats3 = strategy3_three_day_rule(df)
    stats3.update(features)
    stats3['seed'] = seed
    stats3['scenario_id'] = scenario_params['id']
    results.append(stats3)
    
    return results

# 🔥 DOE主循环（8场景×30种子）
def run_full_doe_experiment(seeds: List[int] = list(range(1, 31)), 
                           max_workers: int = 4) -> pd.DataFrame:
    """完整DOE实验"""
    # 8个2^3场景
    scenarios = [
        {'id': 1, 'density': 'low', 'periodicity': 'periodic', 'cluster_level': 'uniform'},
        {'id': 2, 'density': 'low', 'periodicity': 'periodic', 'cluster_level': 'clustered'},
        {'id': 3, 'density': 'low', 'periodicity': 'aperiodic', 'cluster_level': 'uniform'},
        {'id': 4, 'density': 'low', 'periodicity': 'aperiodic', 'cluster_level': 'clustered'},
        {'id': 5, 'density': 'high', 'periodicity': 'periodic', 'cluster_level': 'uniform'},
        {'id': 6, 'density': 'high', 'periodicity': 'periodic', 'cluster_level': 'clustered'},
        {'id': 7, 'density': 'high', 'periodicity': 'aperiodic', 'cluster_level': 'uniform'},
        {'id': 8, 'density': 'high', 'periodicity': 'aperiodic', 'cluster_level': 'clustered'},
    ]
    
    all_results = []
    
    # 并行处理每个场景×种子
    tasks = []
    for scenario in scenarios:
        for seed in seeds:
            tasks.append((scenario, seed))
    
    print(f"🚀 开始DOE: 8场景 × {len(seeds)}种子 = {len(tasks)}实例")
    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(run_single_experiment, scenario, seed): (scenario, seed)
            for scenario, seed in tasks
        }
        
        for future in futures:
            try:
                results = future.result(timeout=300)  # 5分钟超时
                all_results.extend(results)
                scenario, seed = futures[future]
                print(f"✅ 场景{scenario['id']}-{seed}: 完成3策略")
            except Exception as e:
                print(f"❌ 场景{scenario['id']}-{seed}: {e}")
    
    # 保存完整结果
    df_results = pd.DataFrame(all_results)
    
    # 标准列顺序（期刊友好）
    scenario_cols = ['scenario_id', 'density', 'periodicity', 'cluster_level', 
                    'seed', 'strategy', 'density', 'periodicity_cv', 'clustering_tendency']
    metric_cols = ['nb_groups', 'Avg_tasks/group', 'CP_cost', 'Total_cost', 
                  'leakage_reduction_pct', 'total_clustering_cpu_s']  # 你的指标
    
    column_order = scenario_cols + metric_cols
    df_results = df_results.reindex(columns=[c for c in column_order if c in df_results.columns])
    
    filename = 'DOE_3strategies_8scenarios_30seeds.csv'
    df_results.to_csv(filename, index=False, float_format='%.3f')
    
    print(f"\n🎉 DOE完成！结果保存: {filename}")
    print(f"📊 形状: {df_results.shape}")
    print(df_results.groupby(['scenario_id', 'strategy'])['leakage_reduction_pct'].mean().round(2))
    
    return df_results

# 🔥 快速测试单个场景
if __name__ == "__main__":
    # 先测试策略3（需要适配你的calculate_gas_leakage等函数）
    df_test = generate_maintenance_plan_scenario('high', 'periodic', 'clustered', random_seed=42)
    print("测试数据生成OK")
    
    # 运行完整DOE（注释掉测试）
    # doe_results = run_full_doe_experiment(seeds=list(range(1, 6)), max_workers=2)  # 先小规模测试


In [ ]:
# ================================================
# 🔥 CELL 1: 策略1 - ILP最小化组数
# ================================================
import gurobipy as gp
from gurobipy import GRB


def clustering_CR_strategy1(df_window, window_size, window_start):
    """策略1版本：目标是最小化组数 y.sum()"""
    if df_window.empty or window_size < 1:
        print(f"warning: window {window_start} has no tasks")
        return df_window

    N_tasks = df_window.shape[0]
    m = gp.Model('Dynamic_Window_Clustering_Strategy1')
    m.setParam('OutputFlag', 0)
    
    data_pos_num = window_size + 1
    theta = m.addVars(N_tasks, data_pos_num, vtype=GRB.BINARY, name='theta')
    y = m.addVars(data_pos_num, vtype=GRB.BINARY, name='y')  # 🔥 策略1使用y代替cluster，最小化组数

    # 🔥 策略1关键差异：目标函数 = 最小化组数
    m.setObjective(gp.quicksum(y[t] for t in range(data_pos_num)), GRB.MINIMIZE)
    
    # 原版约束完全保留
    m.addConstrs((theta[n, t] <= y[t] for n in range(N_tasks) for t in range(data_pos_num)), 
                 name="cluster_activation")
    m.addConstrs((gp.quicksum(theta[n, t] for t in range(data_pos_num)) == 1 for n in range(N_tasks)), 
                 name="task_assignment")
    m.addConstrs((gp.quicksum(theta[n, t] for n in range(N_tasks)) <= PARAMS['Cmax'] for t in range(data_pos_num)), 
                 name="cluster_capacity")

    # 时间窗口约束（完全复制原版）
    a = [0] * N_tasks
    b = [0] * N_tasks
    for n in range(N_tasks):
        csem = int(df_window.iloc[n]['CSEM'])
        local_time = df_window.iloc[n]['window_time']
        a[n] = max(0, local_time - PARAMS['advanced'])
        alpha = alpha_values[csem]
        b[n] = min(data_pos_num, (local_time + int(0.2 / alpha))-2)

    for n in range(N_tasks):
        for t in range(data_pos_num):
            m.addConstr(t * theta[n,t] + a[n] * (1 - theta[n, t]) >= a[n])
            m.addConstr(t * theta[n,t] + b[n] * (1 - theta[n, t]) <= b[n])

    # 求解参数（同原版）
    m.setParam('LogToConsole', 1)
    m.setParam('OutputFlag', 1)
    m.setParam('TimeLimit', 300)
    m.setParam('MIPGap', 0.05)

    print(f"Window {window_start}: N_tasks={N_tasks} (Strategy1)")
    m.optimize()

    if m.status != GRB.OPTIMAL:
        print(f"Failed to solve window {window_start}: {m.status}")
        return None
    
    # 结果提取（完全同原版）
    centers = {}
    for n in range(N_tasks):
        for t in range(data_pos_num):
            if theta[n, t].x == 1.0:
                centers[n] = t + window_start
                break

    index_n_list = df_window.INDEX.tolist()
    for n in range(len(index_n_list)):
        df_window.loc[df_window['INDEX'] == index_n_list[n], 'CENTER'] = centers.get(n, 0)

    return df_window

def run_strategy1_analysis(params, random_seed):
    """策略1完整分析流程 - 替换clustering_CR为clustering_CR_strategy1"""
    global alpha_values, PARAMS
    PARAMS.update(params)
    
    # 生成维护计划（原版生成器）
    df, alpha_values_list = generate_maintenance_plan(
        N=params['P'], T=params['T'], random_seed=random_seed
    )
    alpha_values = {i: alpha_values_list[i] for i in range(params['P'])}
    
    # 🔥 临时替换聚类函数为策略1版本
    original_clustering_CR = globals().get('clustering_CR', None)
    globals()['clustering_CR'] = clustering_CR_strategy1
    
    # 运行分析（使用替换后的函数）
    result_dict = run_clustering_analysis(params, random_seed)
    
    # 恢复原函数（如果存在）
    if original_clustering_CR:
        globals()['clustering_CR'] = original_clustering_CR
    
    # 计算统计并添加策略标识
    stats = calculate_detailed_stats(result_dict, params)
    stats['strategy'] = 'Strategy1_MinGroups'
    stats['seed'] = random_seed
    
    return stats, result_dict['final_df']

# 🔥 测试并保存策略1结果
print("#" + "="*60)
print("🚀 启动策略1分析 - ILP最小化组数")
print("#" + "="*60)

# 参数配置
PARAMS_STRATEGY1 = {
    'P': 50,      # 沉井数量
    'T': 365,     # 总时间
    'window_size': 60,
    'Cmax': 5,    # 每组最大任务数
    'advanced': 5,
    'Cs': 1000,   # 每次维护成本
    'Calpha': 2000,  # 提前成本系数
    'Cbeta': 1000,   # 延迟成本系数
    'K': 1.0      # 泄漏系数
}

seeds = list(range(1, 6))  # 5个种子测试
strategy1_results = []

for seed in seeds:
    print(f"\n🔄 处理策略1 Seed {seed}...")
    
    stats, df_final = run_strategy1_analysis(PARAMS_STRATEGY1, seed)
    strategy1_results.append(stats)
    
    print(f"✅ Seed{seed}: 组数={stats['nb_groups']}, "
          f"泄漏减少={stats['leakage_reduction_pct']:.1f}%, "
          f"总成本={stats['Total_cost']:.0f}")

# 保存结果
strategy1_df = pd.DataFrame(strategy1_results)
filename = 'strategy1_results.csv'
strategy1_df.to_csv(filename, index=False, float_format='%.3f')

print(f"\n🎉 策略1完成！结果保存到: {filename}")
print(f"📊 统计: {len(strategy1_df)} 种子, 平均组数={strategy1_df['nb_groups'].mean():.1f}")
print(f"💾 文件包含所有详细指标，可直接用于对比分析")

# 显示摘要表格
print("\n📋 策略1结果摘要:")
display(strategy1_df[['seed', 'nb_groups', 'Total_cost', 'leakage_reduction_pct', 'total_clustering_cpu_s']].round(2))


In [ ]:
# ================================================
# 🔥 CELL 2: 策略2 - 利润最大化 + 维护间隔约束
# ================================================

def clustering_CR_strategy2(df_window, window_size, window_start, full_df=None, D_min_dict=None, D_max_dict=None):
    """策略2版本：最大化利润(setup_cost - CP_total) + 维护间隔约束"""
    if df_window.empty or window_size < 1:
        print(f"warning: window {window_start} has no tasks")
        return df_window

    N = df_window.shape[0]
    mdl = gp.Model('ProfitMax_Strategy2')
    mdl.setParam('OutputFlag', 0)

    data_pos_num = window_size + 1
    theta = mdl.addVars(N, data_pos_num, vtype=GRB.BINARY, name='theta')
    y = mdl.addVars(data_pos_num, vtype=GRB.BINARY, name='y')

    # 🔥 策略2目标函数：最大化利润 = setup节省 - CP成本
    setup_savings = PARAMS['Cs'] * (theta.sum() - y.sum())
    
    # CP成本矩阵
    CP_cost = 0
    for n in range(N):
        local_time_n = int(df_window.iloc[n]['window_time'])
        csem_n = int(df_window.iloc[n]['CSEM'])
        alpha_n = alpha_values[csem_n]
        
        for t in range(data_pos_num):
            deviation = t - local_time_n
            if deviation > 0:
                cp_cost = PARAMS['Cbeta'] * deviation
            elif deviation < 0:
                cp_cost = PARAMS['Calpha'] * (-deviation)
            else:
                cp_cost = 0
            CP_cost += cp_cost * theta[n, t]
    
    obj = setup_savings - CP_cost
    mdl.setObjective(obj, GRB.MAXIMIZE)

    # 原版约束完全保留
    # 任务分配约束
    mdl.addConstrs((theta.sum(n, '*') == 1 for n in range(N)), name="task_assignment")
    
    # 容量约束  
    mdl.addConstrs((theta.sum('*', t) <= PARAMS['Cmax'] for t in range(data_pos_num)), name="cluster_capacity")
    
    # 聚类激活约束
    M = PARAMS['Cmax']
    mdl.addConstrs((theta.sum('*', t) + M * (1 - y[t]) >= 1 for t in range(data_pos_num)), name="activate_lb")
    mdl.addConstrs((theta.sum('*', t) <= M * y[t] for t in range(data_pos_num)), name="activate_ub")
    
    # 时间窗口约束
    a = [0] * N
    b = [0] * N
    for n in range(N):
        local_time = int(df_window.iloc[n]['window_time'])
        a[n] = max(0, local_time - PARAMS['advanced'])
        b[n] = min(data_pos_num, (local_time + int(0.2 / alpha_values[int(df_window.iloc[n]["CSEM"])]))-2)

    for n in range(N):
        for t in range(data_pos_num):
            mdl.addConstr(t * theta[n,t] + a[n] * (1 - theta[n, t]) >= a[n])
            mdl.addConstr(t * theta[n,t] + b[n] * (1 - theta[n, t]) <= b[n])

    # 🔥 策略2关键特性：维护间隔约束
    if full_df is not None and D_min_dict is not None and window_start > 0:
        for n in range(N):
            row = df_window.iloc[n]
            caisson = int(row['CSEM'])
            
            prev_rows = full_df[
                (full_df['CSEM'] == caisson) & 
                (full_df['CENTER'] < row['DATE']) &
                (full_df['OPERATION'] == 'x')
            ].sort_values('CENTER')
            
            if not prev_rows.empty:
                prev_center = int(prev_rows.iloc[-1]['CENTER'])
                t_now_local = gp.quicksum(t * theta[n, t] for t in range(data_pos_num))
                d_min_i = D_min_dict.get(caisson, PARAMS['window_size'] * 0.6)
                d_max_i = D_max_dict.get(caisson, PARAMS['window_size'] * 1.4)
                
                mdl.addConstr(t_now_local + window_start - prev_center >= d_min_i, name=f"dmin_n{n}")
                mdl.addConstr(t_now_local + window_start - prev_center <= d_max_i, name=f"dmax_n{n}")

    # 求解参数
    mdl.setParam('LogToConsole', 1)
    mdl.setParam('OutputFlag', 1)
    mdl.setParam('TimeLimit', 300)
    mdl.setParam('MIPGap', 0.05)

    print(f"Window {window_start}: N_tasks={N} (Strategy2)")
    mdl.optimize()

    if mdl.status != GRB.OPTIMAL:
        print(f"Failed to solve window {window_start}: {mdl.status}")
        return None
    
    # 结果提取（同原版）
    centers = {}
    for n in range(N):
        for t in range(data_pos_num):
            if theta[n, t].x == 1.0:
                centers[n] = t + window_start
                break

    index_n_list = df_window.INDEX.tolist()
    for n in range(len(index_n_list)):
        df_window.loc[df_window['INDEX'] == index_n_list[n], 'CENTER'] = centers.get(n, 0)

    print(f"Strategy2 Obj: {mdl.ObjVal:.1f}")
    return df_window

def calculate_maintenance_period_from_data(df, d_min_ratio=0.6, d_max_ratio=1.4):
    """从数据中提取每个caisson的维修周期约束"""
    maintenance_periods = {}
    D_min_dict = {}
    D_max_dict = {}
    
    print("\n🔍 策略2 - 提取维修周期约束:")
    print(f"{'CSEM':<8} {'任务数':<8} {'周期(days)':<12} {'D_min':<8} {'D_max':<8}")
    print("-" * 50)
    
    for csem_id in sorted(df['CSEM'].unique()):
        csem_tasks = df[(df['CSEM'] == csem_id) & (df['OPERATION'] == 'x')].sort_values('DATE')
        
        if len(csem_tasks) < 2:
            period = PARAMS['window_size']
        else:
            intervals = csem_tasks['DATE'].diff().dropna()
            period = intervals.mean()
        
        d_min = int(np.ceil(period * d_min_ratio))
        d_max = int(np.ceil(period * d_max_ratio))
        
        maintenance_periods[csem_id] = period
        D_min_dict[csem_id] = d_min
        D_max_dict[csem_id] = d_max
        
        print(f"{csem_id:<8} {len(csem_tasks):<8} {period:<12.1f} {d_min:<8} {d_max:<8}")
    
    return maintenance_periods, D_min_dict, D_max_dict

def run_strategy2_analysis(params, random_seed):
    """策略2完整分析流程：利润最大化 + 维护间隔约束"""
    global alpha_values, PARAMS
    PARAMS.update(params)
    
    # 生成数据
    df, alpha_values_list = generate_maintenance_plan(
        N=params['P'], T=params['T'], random_seed=random_seed
    )
    alpha_values = {i: alpha_values_list[i] for i in range(params['P'])}
    
    # 🔥 策略2特性1：计算维护周期约束
    maintenance_periods, D_min_dict, D_max_dict = calculate_maintenance_period_from_data(df)
    
    # 🔥 临时替换聚类函数为策略2
    original_clustering_CR = globals().get('clustering_CR', None)
    globals()['clustering_CR'] = clustering_CR_strategy2
    
    # 运行分析（自动使用维护约束）
    result_dict = run_clustering_analysis(params, random_seed)
    
    # 恢复原函数
    if original_clustering_CR:
        globals()['clustering_CR'] = original_clustering_CR
    
    # 计算统计并标记策略
    stats = calculate_detailed_stats(result_dict, params)
    stats['strategy'] = 'Strategy2_ProfitMax'
    stats['seed'] = random_seed
    stats['maintenance_constraints'] = len(D_min_dict)
    
    return stats, result_dict['final_df'], D_min_dict

# 🔥 测试并保存策略2结果
print("#" + "="*60)
print("🚀 启动策略2分析 - 利润最大化 + 维护间隔约束")
print("#" + "="*60)

# 参数配置（增加利润导向参数）
PARAMS_STRATEGY2 = {
    'P': 50,        # 沉井数量
    'T': 365,       # 总时间天数
    'window_size': 60,
    'Cmax': 5,      # 每组最大任务数
    'advanced': 5,  # 提前天数限制
    'Cs': 1000,     # 每次setup成本
    'Calpha': 2000, # 提前惩罚系数
    'Cbeta': 1000,  # 延迟惩罚系数
    'K': 1.0        # 泄漏系数
}

seeds = list(range(1, 6))  # 5个种子测试
strategy2_results = []

for seed in seeds:
    print(f"\n🔄 处理策略2 Seed {seed}...")
    
    stats, df_final, d_min_dict = run_strategy2_analysis(PARAMS_STRATEGY2, seed)
    strategy2_results.append(stats)
    
    print(f"✅ Seed{seed}: 组数={stats['nb_groups']:.0f}, "
          f"总成本={stats['Total_cost']:.0f}, "
          f"泄漏减少={stats['leakage_reduction_pct']:.1f}%")

# 保存结果
strategy2_df = pd.DataFrame(strategy2_results)
filename = 'strategy2_results.csv'
strategy2_df.to_csv(filename, index=False, float_format='%.3f')

print(f"\n🎉 策略2完成！结果保存到: {filename}")
print(f"📊 统计: {len(strategy2_df)} 种子, 平均组数={strategy2_df['nb_groups'].mean():.1f}")
print(f"💾 包含利润最大化 + 维护间隔约束的所有指标")

# 显示对比摘要
print("\n📋 策略2结果摘要:")
summary_cols = ['seed', 'nb_groups', 'Total_cost', 'CP_cost', 'leakage_reduction_pct', 'total_clustering_cpu_s']
display(strategy2_df[summary_cols].round(2))

print("\n✅ Cell2执行完毕 - 策略2结果已准备好与策略1对比！")


In [ ]:
# ================================================
# 🔥 CELL 3: 策略3 - 3天简单聚合（无Gurobi）
# ================================================

def clustering_CR_strategy3(df_window, window_size, window_start):
    """
    策略3：3天固定块聚合（超快！）
    - 每3天一个块：第1天≤Cmax，第2天≤Cmax，第3天兜底
    - CPU时间<0.01s/window，完全取代Gurobi
    """
    if df_window.empty or window_size < 1:
        print(f"warning: window {window_start} has no tasks")
        return df_window
    
    print(f"Window {window_start}: N_tasks={len(df_window)} (Strategy3)")
    
    # 初始化CENTER为原始时间
    df_window = df_window.sort_values('window_time').copy()
    df_window['CENTER'] = df_window['DATE']
    
    # 🔥 3天固定块聚合逻辑
    block_size = 3  # 策略3核心参数
    for day_block in range(0, window_size, block_size):
        block_start = window_start + day_block
        
        # 该3天块的任务
        mask_block = (df_window['window_time'] >= day_block) & \
                    (df_window['window_time'] < day_block + block_size)
        block_tasks = df_window[mask_block].index.tolist()
        
        if not block_tasks:
            continue
        
        # 优先第1天聚合（≤Cmax）
        day1_tasks = block_tasks[:PARAMS['Cmax']]
        df_window.loc[day1_tasks, 'CENTER'] = block_start
        
        # 剩余→第2天
        day2_start_idx = PARAMS['Cmax']
        day2_tasks = block_tasks[day2_start_idx:day2_start_idx + PARAMS['Cmax']]
        if day2_tasks:
            df_window.loc[day2_tasks, 'CENTER'] = block_start + 1
        
        # 最后剩余→第3天
        day3_tasks = block_tasks[day2_start_idx + PARAMS['Cmax']:]
        if day3_tasks:
            df_window.loc[day3_tasks, 'CENTER'] = block_start + 2
    
    unique_centers = df_window['CENTER'].nunique()
    print(f"  → Strategy3完成: {unique_centers} 聚类点, 速度⚡")
    return df_window

def run_strategy3_analysis(params, random_seed):
    """策略3完整分析：3天简单聚合（无优化器）"""
    global alpha_values, PARAMS
    PARAMS.update(params)
    
    # 生成数据（同原版）
    df, alpha_values_list = generate_maintenance_plan(
        N=params['P'], T=params['T'], random_seed=random_seed
    )
    alpha_values = {i: alpha_values_list[i] for i in range(params['P'])}
    
    print(f"Generated {len(df)} tasks for seed {random_seed}")
    
    # 🔥 临时替换聚类函数为策略3（无需Gurobi！）
    original_clustering_CR = globals().get('clustering_CR', None)
    globals()['clustering_CR'] = clustering_CR_strategy3
    
    # 直接复用原版主流程
    result_dict = run_clustering_analysis(params, random_seed)
    
    # 恢复原函数
    if original_clustering_CR:
        globals()['clustering_CR'] = original_clustering_CR
    
    # 计算统计 + 策略标记
    stats = calculate_detailed_stats(result_dict, params)
    stats['strategy'] = 'Strategy3_Simple3Day'
    stats['seed'] = random_seed
    stats['method'] = 'NoOptimizer'
    stats['speedup_factor'] = '1000x'  # 相对于Gurobi
    
    return stats, result_dict['final_df']

# 🔥 测试并保存策略3结果
print("#" + "="*70)
print("⚡ 启动策略3分析 - 3天简单聚合（无Gurobi，超快！）")
print("#" + "="*70)

# 参数配置（相同场景）
PARAMS_STRATEGY3 = {
    'P': 50,        # 沉井数量
    'T': 365,       # 总时间
    'window_size': 60,
    'Cmax': 5,      # 每3天块最多5个任务/天
    'advanced': 5,
    'Cs': 1000,
    'Calpha': 2000,
    'Cbeta': 1000,
    'K': 1.0
}

seeds = list(range(1, 6))  # 5个种子，预计<1秒完成！
strategy3_results = []
start_total = time.time()

for seed in seeds:
    print(f"\n⚡ Seed {seed} (预计0.1s)...")
    t1 = time.time()
    
    stats, df_final = run_strategy3_analysis(PARAMS_STRATEGY3, seed)
    strategy3_results.append(stats)
    
    elapsed = time.time() - t1
    print(f"✅ Seed{seed}: {elapsed*1000:.0f}ms | "
          f"组数={stats['nb_groups']:.0f} | "
          f"成本={stats['Total_cost']:.0f} | "
          f"泄漏↓{stats['leakage_reduction_pct']:.1f}%")

total_time = time.time() - start_total
print(f"\n🚀 策略3全5种子: {total_time:.2f}s ({total_time/5:.2f}s/seed)")

# 保存结果
strategy3_df = pd.DataFrame(strategy3_results)
filename = 'strategy3_results.csv'
strategy3_df.to_csv(filename, index=False, float_format='%.3f')

print(f"\n🎉 策略3完成！结果保存到: {filename}")
print(f"📊 平均: 组数={strategy3_df['nb_groups'].mean():.1f}, "
      f"总成本={strategy3_df['Total_cost'].mean():.0f}")

# 🔥 三策略对比摘要表
print("\n" + "="*80)
print("📊 三策略快速对比 (平均值):")
print("="*80)

# 加载前两个策略结果（假设已运行）
try:
    s1_df = pd.read_csv('strategy1_results.csv')
    s2_df = pd.read_csv('strategy2_results.csv')
    comparison_data = {
        '策略': ['Strategy1(最小组数)', 'Strategy2(利润最大化)', 'Strategy3(3天简单)'],
        '组数': [s1_df['nb_groups'].mean(), s2_df['nb_groups'].mean(), strategy3_df['nb_groups'].mean()],
        '总成本': [s1_df['Total_cost'].mean(), s2_df['Total_cost'].mean(), strategy3_df['Total_cost'].mean()],
        '泄漏减少%': [s1_df['leakage_reduction_pct'].mean(), s2_df['leakage_reduction_pct'].mean(), 
                     strategy3_df['leakage_reduction_pct'].mean()],
        'CPU时间(s)': [s1_df['total_clustering_cpu_s'].mean(), s2_df['total_clustering_cpu_s'].mean(), 
                      strategy3_df['total_clustering_cpu_s'].mean()],
        '速度': ['慢', '中', '⚡超快']
    }
    comparison_df = pd.DataFrame(comparison_data)
    display(comparison_df.round(1))
except:
    print("📋 Strategy1/2结果未找到，跳过对比")
    display(strategy3_df[['seed', 'nb_groups', 'Total_cost', 'leakage_reduction_pct', 'total_clustering_cpu_s']].round(2))

print("\n✅ Cell3执行完毕 - 策略3极速完成，可与ILP策略对比效果！")


In [ ]:
# ================================================
# 🔥 CELL 4: 结果汇总 + DOE统计分析
# ================================================

# 加载3个策略结果
df1 = pd.read_csv('strategy1_results.csv')
df2 = pd.read_csv('strategy2_results.csv')
df3 = pd.read_csv('strategy3_results.csv')

# 合并（假设每个都有scenario特征列）
all_results = pd.concat([df1, df2, df3], ignore_index=True)

# 🔥 关键汇总表：每个场景各策略平均性能
summary_table = all_results.groupby(['scenario_id', 'strategy'])[
    ['leakage_reduction_pct', 'Total_cost', 'nb_groups', 'total_clustering_cpu_s']
].mean().round(2).reset_index()

print("📊 DOE汇总表（8场景×3策略）:")
print(summary_table.pivot(index='scenario_id', columns='strategy', 
                         values='leakage_reduction_pct'))

# 🔥 排名分析：每个场景哪个策略最优
def rank_strategies(group):
    leakage_rank = group.sort_values('leakage_reduction_pct', ascending=False)['strategy'].tolist()
    return pd.Series({'best_leakage': leakage_rank[0]})

scenario_ranking = all_results.groupby('scenario_id').apply(rank_strategies)
print("\n🏆 各场景泄漏减少最优策略:")
print(scenario_ranking)

# 🔥 保存最终分析结果
summary_table.to_csv('doe_summary_analysis.csv', index=False)
all_results.to_csv('all_strategies_full_results.csv', index=False)

print("\n🎉 完整DOE分析完成！")
print("📁 输出文件:")
print("- doe_summary_analysis.csv (关键汇总表)")
print("- all_strategies_full_results.csv (240行完整数据)")
print("- 每个策略单独CSV (策略1/2/3_results.csv)")
